# 03 - Data and Embeddings

Turn the aircraft maintenance manuals into a searchable knowledge graph. `SimpleKGPipeline` from `neo4j-graphrag` chunks each manual, embeds the chunks, extracts `OperatingLimit` entities, and resolves duplicates in a single pass.

```
(:Document) <-[:FROM_DOCUMENT]- (:Chunk) -[:NEXT_CHUNK]-> (:Chunk)
                                    |
                              (embedding vector)
                              (extracted OperatingLimit entities)
```

This notebook loads **two manuals** (A320-200 / V2500 and B737-800 / CFM56), so the retrievers in notebooks 04 and 05 have more than one aircraft model to discriminate between.

## Prerequisites
- **01_etl_to_neo4j** has loaded the aircraft topology
- Running in a Databricks workspace with Foundation Model APIs enabled (BGE-large embeddings, Llama 3.3 70B)
- `pip install neo4j-graphrag mlflow`

## What you'll learn
- Build the Document -> Chunk structure with embeddings
- Create vector and fulltext indexes
- Cross-link extracted entities to the aircraft topology
- Run semantic search over maintenance procedures

## Section 1: Configuration

In [ ]:
%pip install neo4j-graphrag mlflow

In [ ]:
# ============================================
# CONFIGURATION
# ============================================

NEO4J_URI = ""
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = ""

# Where to read the manuals from: "github" (default), "local", or "volume"
DATA_SOURCE = "github"

if not NEO4J_URI or not NEO4J_PASSWORD:
    print("WARNING: enter your Neo4j credentials above before running!")
else:
    print(f"Configuration ready - {NEO4J_URI} (DATA_SOURCE={DATA_SOURCE})")

In [ ]:
from neo4j_graphrag.indexes import create_vector_index, create_fulltext_index

from data_utils import (
    Neo4jConnection, load_text, get_embedder, get_llm,
    run_pipeline, EMBEDDING_DIMENSIONS,
)

## Maintenance manuals

Two manuals, each linked to its aircraft model. The `aircraftType` matches `Aircraft.model` from notebook 01, so the cross-link in Part 3 connects each document to its five aircraft.

In [ ]:
MANUALS = [
    {
        "filename": "MAINTENANCE_A320.md",
        "documentId": "AMM-A320-2024-001",
        "aircraftType": "A320-200",
        "title": "A320-200 Maintenance and Troubleshooting Manual",
    },
    {
        "filename": "MAINTENANCE_B737.md",
        "documentId": "AMM-B737-2024-001",
        "aircraftType": "B737-800",
        "title": "B737-800 Maintenance and Troubleshooting Manual",
    },
]
for m in MANUALS:
    print(f"{m['aircraftType']:<10} {m['filename']}")

## Connect to Neo4j

The database should already hold the aircraft topology from notebook 01.

In [ ]:
neo4j = Neo4jConnection(NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD).verify()
driver = neo4j.driver
neo4j.get_graph_stats()

## Clear previous enrichment (optional)

Remove Document, Chunk, and OperatingLimit nodes from earlier runs. The aircraft topology is preserved.

In [ ]:
neo4j.clear_chunks()

## Initialize LLM and embedder

Both run on Databricks Foundation Model APIs: Llama 3.3 70B for entity extraction, BGE-large for chunk embeddings.

In [ ]:
llm = get_llm()
embedder = get_embedder()
print(f"LLM:      {llm.model_id}")
print(f"Embedder: {embedder.model_id}")

---

# Part 1: Run SimpleKGPipeline

For each manual the pipeline:
1. **Splits** the text into 800-character chunks with 100-character overlap
2. **Embeds** each chunk (1024-dimensional BGE vectors)
3. **Extracts** `OperatingLimit` entities with the LLM
4. **Resolves** duplicate entities across chunks

A `ContextPrependingSplitter` prepends `[DOCUMENT CONTEXT] Aircraft Type: ...` to every chunk so the LLM never confuses an engine designation (V2500, CFM56-7B) for the aircraft type.

> This cell takes a few minutes per manual depending on LLM response time.

In [ ]:
for manual in MANUALS:
    text = load_text(manual["filename"], DATA_SOURCE)
    print(f"\n=== {manual['filename']} ({len(text):,} chars) ===")
    run_pipeline(
        driver=driver,
        llm=llm,
        embedder=embedder,
        text=text,
        document_metadata={
            "documentId": manual["documentId"],
            "aircraftType": manual["aircraftType"],
            "title": manual["title"],
            "type": "maintenance_manual",
        },
        context=(
            f"[DOCUMENT CONTEXT] Aircraft Type: {manual['aircraftType']} "
            f"| Title: {manual['title']}\n\n"
        ),
    )

## Inspect the knowledge graph

In [ ]:
neo4j.get_graph_stats()

records, _, _ = driver.execute_query("""
    MATCH (d:Document)
    OPTIONAL MATCH (d)<-[:FROM_DOCUMENT]-(c:Chunk)
    RETURN d.documentId AS document_id, d.title AS title, count(c) AS chunks
    ORDER BY document_id
""")
print("\n=== Documents and chunks ===")
for r in records:
    print(f"  {r['document_id']}: {r['chunks']} chunks  ({r['title']})")

## Inspect extracted entities

The LLM extracted `OperatingLimit` entities (EGT thresholds, vibration limits, etc.), qualified by aircraft type so limits from the two manuals stay distinct.

In [ ]:
records, _, _ = driver.execute_query("""
    MATCH (ol:OperatingLimit)
    RETURN ol.name AS name, ol.parameterName AS param,
           ol.aircraftType AS aircraft, ol.unit AS unit,
           ol.maxValue AS max_value, ol.regime AS regime
    ORDER BY ol.aircraftType, ol.name
""")
print(f"Extracted {len(records)} OperatingLimit entities:\n")
for r in records:
    regime = f" ({r['regime']})" if r['regime'] else ""
    print(f"  {r['name']}: max={r['max_value'] or 'N/A'} {r['unit'] or ''}{regime}")

---

# Part 2: Create indexes for retrieval

- **Vector index** for semantic similarity (cosine over 1024-dim embeddings)
- **Fulltext index** for keyword search (powers the `HybridRetriever` in notebook 05)

In [ ]:
INDEX_NAME = "maintenanceChunkEmbeddings"
create_vector_index(
    driver=driver, name=INDEX_NAME, label="Chunk",
    embedding_property="embedding", dimensions=EMBEDDING_DIMENSIONS, similarity_fn="cosine",
)
print(f"Created vector index: {INDEX_NAME} ({EMBEDDING_DIMENSIONS} dims, cosine)")

In [ ]:
FULLTEXT_INDEX_NAME = "maintenanceChunkText"
create_fulltext_index(
    driver=driver, name=FULLTEXT_INDEX_NAME, label="Chunk", node_properties=["text"],
)
print(f"Created fulltext index: {FULLTEXT_INDEX_NAME}")

---

# Part 3: Cross-link to the aircraft topology

Two links bridge the manuals to the topology from notebook 01:

1. `Document -[:APPLIES_TO]-> Aircraft` - the document `aircraftType` matches `Aircraft.model`
2. `Sensor -[:HAS_LIMIT]-> OperatingLimit` - the limit `parameterName` matches `Sensor.type`

```
Chunk -[:FROM_DOCUMENT]-> Document -[:APPLIES_TO]-> Aircraft
                                                      |
                                            [:HAS_SYSTEM]-> System -[:HAS_SENSOR]-> Sensor
                                                                                      |
                                                                            [:HAS_LIMIT]-> OperatingLimit
```

In [ ]:
records, _, _ = driver.execute_query("""
    MATCH (d:Document) WHERE d.aircraftType IS NOT NULL
    MATCH (a:Aircraft {model: d.aircraftType})
    MERGE (d)-[:APPLIES_TO]->(a)
    RETURN d.documentId AS doc, d.aircraftType AS model, count(a) AS aircraft
""")
print("Document -[:APPLIES_TO]-> Aircraft:")
for r in records:
    print(f"  {r['doc']} -> {r['model']} ({r['aircraft']} aircraft)")

records, _, _ = driver.execute_query("""
    MATCH (a:Aircraft)-[:HAS_SYSTEM]->(:System)-[:HAS_SENSOR]->(s:Sensor)
    MATCH (ol:OperatingLimit {parameterName: s.type, aircraftType: a.model})
    MERGE (s)-[:HAS_LIMIT]->(ol)
    RETURN count(DISTINCT ol) AS limits_linked
""")
print(f"\nSensor -[:HAS_LIMIT]-> OperatingLimit: {records[0]['limits_linked']} limits linked")
if records[0]["limits_linked"] == 0:
    print("  (depends on which limits the LLM extracted)")

In [ ]:
records, _, _ = driver.execute_query("""
    MATCH (c:Chunk)-[:FROM_DOCUMENT]->(d:Document)-[:APPLIES_TO]->(a:Aircraft)
    WITH a, count(DISTINCT c) AS chunks
    OPTIONAL MATCH (a)-[:HAS_SYSTEM]->(:System)-[:HAS_SENSOR]->(:Sensor)-[:HAS_LIMIT]->(ol:OperatingLimit)
    RETURN a.model AS model, chunks, count(DISTINCT ol) AS operating_limits
    ORDER BY model
""")
print(f"{'Model':<12} {'Chunks':>8} {'Limits':>8}")
print("-" * 30)
for r in records:
    print(f"{r['model']:<12} {r['chunks']:>8} {r['operating_limits']:>8}")

---

# Part 4: Test semantic search

Embed a query, find the closest chunks by cosine distance, and return them ranked by score.

In [ ]:
def vector_search(query, top_k=3):
    embedding = embedder.embed_query(query)
    records, _, _ = driver.execute_query("""
        CALL db.index.vector.queryNodes($index_name, $top_k, $embedding)
        YIELD node, score
        RETURN node.text AS text, node.index AS idx, score
    """, index_name=INDEX_NAME, top_k=top_k, embedding=embedding)
    return records

query = "How do I troubleshoot engine vibration?"
print(f'Query: "{query}"\n' + "=" * 70)
for i, r in enumerate(vector_search(query)):
    print(f"\n[{i+1}] score={r['score']:.4f} (chunk {r['idx']})")
    print(f"    {r['text'][:220]}...")

## Compare queries

Semantic search finds relevant procedures even when the wording differs from the manual.

In [ ]:
queries = [
    "What are the EGT limits during takeoff?",
    "How to detect bearing wear in the engine?",
    "What causes hydraulic pressure loss?",
    "When should I replace the fuel filter?",
]
for query in queries:
    print(f'\nQuery: "{query}"')
    print("-" * 60)
    r = vector_search(query, top_k=1)[0]
    print(f"Best match (score {r['score']:.4f}):\n  {r['text'][:200]}...")

## Summary

You built the foundation for semantic search over two aircraft maintenance manuals:

1. **Document -> Chunk** structure with embeddings, via `SimpleKGPipeline`
2. **Vector and fulltext indexes** (`maintenanceChunkEmbeddings`, `maintenanceChunkText`)
3. **Cross-links** to the topology: `APPLIES_TO` and `HAS_LIMIT`
4. **Semantic search** verified

**Next:** **04_graphrag_retrievers** builds retrieval strategies that combine vector search with graph traversal. **05_hybrid_retrievers** adds keyword precision.

In [ ]:
neo4j.close()